# Baseline: module health and bottleneck analysis

This notebook checks whether each pipeline stage is connected and measures likely bottlenecks between modules. It does not replace production code in `src/`; it calls the existing modules and reads generated artifacts.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    import psutil
except ImportError:
    psutil = None

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "local.yaml"
RUN_ID = "smoke"
LIMIT_SAMPLES = 4
RUN_SMOKE_COMMANDS = False  # Set true to execute short end-to-end commands from this notebook.

pd.set_option("display.max_columns", 100)


In [2]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.feature_eng_pipeline import load_config as load_preprocess_config, read_jsonl
from src.pipelines.feature_extraction_pipeline import load_config as load_feature_config
from src.pipelines.latent_dit_pipeline import (
    FeatureLatentDataset,
    build_flow_matching_sampler,
    compute_latent_stats,
    load_feature_records,
    load_pipeline_config,
    make_loader_kwargs,
)
from src.pipelines.training_pipeline import build_model, infer_latent_dit_shape


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
preprocess_config = load_preprocess_config(CONFIG_PATH)
feature_config = load_feature_config(CONFIG_PATH)
latent_config = load_pipeline_config(CONFIG_PATH)

connectivity = pd.DataFrame([
    {
        "check": "processed root feeds feature extraction",
        "expected": str(preprocess_config.processed_root),
        "actual": str(feature_config.processed_root),
        "ok": preprocess_config.processed_root == feature_config.processed_root,
    },
    {
        "check": "train feature index feeds training",
        "expected": str(feature_config.feature_index_path("train")),
        "actual": str(latent_config.training.feature_index),
        "ok": feature_config.feature_index_path("train") == latent_config.training.feature_index,
    },
    {
        "check": "test feature index feeds inference",
        "expected": str(feature_config.feature_index_path("test")),
        "actual": str(latent_config.inference.feature_index),
        "ok": feature_config.feature_index_path("test") == latent_config.inference.feature_index,
    },
    {
        "check": "inference writes 04_predictions",
        "expected": str(PROJECT_ROOT / "data" / "04_predictions" / latent_config.dataset_name),
        "actual": str(latent_config.inference.output_root),
        "ok": latent_config.inference.output_root == PROJECT_ROOT / "data" / "04_predictions" / latent_config.dataset_name
              or latent_config.inference.output_root == Path("data/04_predictions") / latent_config.dataset_name,
    },
])
connectivity


,check,expected,actual,ok
0,processed root feeds feature extraction,data/02_processed/shanghaitech,data/02_processed/shanghaitech,True
1,train feature index feeds training,data/03_features/shanghaitech/train_feature_in...,data/03_features/shanghaitech/train_feature_in...,True
2,test feature index feeds inference,data/03_features/shanghaitech/test_feature_ind...,data/03_features/shanghaitech/test_feature_ind...,True
3,inference writes 04_predictions,/workspace/VAD/data/04_predictions/shanghaitech,data/04_predictions/shanghaitech,True


In [4]:
def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024) if path.is_file() else 0.0


def count_files(path: Path, pattern="*") -> int:
    return sum(1 for _ in path.rglob(pattern)) if path.is_dir() else 0

artifact_rows = [
    {"artifact": "processed_root", "path": preprocess_config.processed_root, "exists": preprocess_config.processed_root.exists(), "files": count_files(preprocess_config.processed_root)},
    {"artifact": "train_samples", "path": preprocess_config.train_samples_path, "exists": preprocess_config.train_samples_path.is_file(), "size_mb": file_size_mb(preprocess_config.train_samples_path)},
    {"artifact": "test_samples", "path": preprocess_config.test_samples_path, "exists": preprocess_config.test_samples_path.is_file(), "size_mb": file_size_mb(preprocess_config.test_samples_path)},
    {"artifact": "feature_root", "path": feature_config.feature_root, "exists": feature_config.feature_root.exists(), "files": count_files(feature_config.feature_root)},
    {"artifact": "train_feature_index", "path": latent_config.training.feature_index, "exists": latent_config.training.feature_index.is_file(), "size_mb": file_size_mb(latent_config.training.feature_index)},
    {"artifact": "test_feature_index", "path": latent_config.inference.feature_index, "exists": latent_config.inference.feature_index.is_file(), "size_mb": file_size_mb(latent_config.inference.feature_index)},
    {"artifact": "prediction_root", "path": latent_config.inference.output_root, "exists": latent_config.inference.output_root.exists(), "files": count_files(latent_config.inference.output_root)},
]
artifacts = pd.DataFrame(artifact_rows)
artifacts["path"] = artifacts["path"].astype(str)
artifacts


,artifact,path,exists,files,size_mb
0,processed_root,data/02_processed/shanghaitech,False,0.0,NaN
1,train_samples,data/02_processed/shanghaitech/samples_train.j...,False,NaN,0.0
2,test_samples,data/02_processed/shanghaitech/samples_test.jsonl,False,NaN,0.0
3,feature_root,data/03_features/shanghaitech,False,0.0,NaN
4,train_feature_index,data/03_features/shanghaitech/train_feature_in...,False,NaN,0.0
5,test_feature_index,data/03_features/shanghaitech/test_feature_ind...,False,NaN,0.0
6,prediction_root,data/04_predictions/shanghaitech,False,0.0,NaN


In [5]:
def summarize_jsonl(path: Path, name: str):
    records = read_jsonl(path) if path.is_file() else []
    if not records:
        return {"name": name, "rows": 0}
    df = pd.DataFrame(records)
    summary = {"name": name, "rows": len(df), "columns": list(df.columns)}
    for column in ["video_id", "split", "future_label"]:
        if column in df:
            summary[f"{column}_unique"] = df[column].nunique()
    return summary

sample_summaries = pd.DataFrame([
    summarize_jsonl(preprocess_config.train_samples_path, "processed train samples"),
    summarize_jsonl(preprocess_config.test_samples_path, "processed test samples"),
    summarize_jsonl(latent_config.training.feature_index, "train feature index"),
    summarize_jsonl(latent_config.inference.feature_index, "test feature index"),
])
sample_summaries


,name,rows
0,processed train samples,0
1,processed test samples,0
2,train feature index,0
3,test feature index,0


In [6]:
def load_available_records(path: Path, normal_only=False, limit=LIMIT_SAMPLES):
    if not path.is_file():
        return []
    try:
        return load_feature_records(path, normal_only=normal_only, limit_samples=limit)
    except Exception as exc:
        print(f"Could not load {path}: {exc}")
        return []

train_records = load_available_records(latent_config.training.feature_index, normal_only=True)
test_records = load_available_records(latent_config.inference.feature_index, normal_only=False)

if train_records:
    dataset = FeatureLatentDataset(train_records)
    sample = dataset[0]
    context_shape = tuple(sample["context"].shape)
    future_shape = tuple(sample["future"].shape)
    print("context E_c shape [T_c, N_c, D_c]:", context_shape)
    print("future E_v shape [T_f, C, H, W]:", future_shape)
    print("sample metadata keys:", sorted(sample["metadata"].keys()))
else:
    print("No cached train features found. Run extract_features first.")


No cached train features found. Run extract_features first.


In [7]:
def benchmark_loader(records, batch_size=4, num_workers=0, max_batches=20):
    if not records:
        return {"batches": 0, "samples": 0, "seconds": np.nan, "samples_per_sec": np.nan}
    dataset = FeatureLatentDataset(records)
    loader = torch.utils.data.DataLoader(dataset, **make_loader_kwargs(batch_size, num_workers, shuffle=False))
    start = time.perf_counter()
    batches = 0
    samples = 0
    for batch in loader:
        batches += 1
        samples += batch["context"].shape[0]
        if batches >= max_batches:
            break
    elapsed = time.perf_counter() - start
    return {
        "batches": batches,
        "samples": samples,
        "seconds": elapsed,
        "samples_per_sec": samples / elapsed if elapsed > 0 else np.nan,
        "batch_size": batch_size,
        "num_workers": num_workers,
    }

loader_benchmarks = pd.DataFrame([
    benchmark_loader(train_records, batch_size=latent_config.training.batch_size, num_workers=0),
    benchmark_loader(train_records, batch_size=latent_config.training.batch_size, num_workers=min(2, os.cpu_count() or 1)),
])
loader_benchmarks


,batches,samples,seconds,samples_per_sec
0,0,0,NaN,NaN
1,0,0,NaN,NaN


In [8]:
def benchmark_model_forward(records, repeats=5):
    if not records:
        return {"available": False}
    dataset = FeatureLatentDataset(records)
    batch = next(iter(torch.utils.data.DataLoader(dataset, **make_loader_kwargs(min(2, len(dataset)), 0, shuffle=False))))
    shape = infer_latent_dit_shape(dataset[0])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(latent_config.model, shape).to(device).eval()
    flow = build_flow_matching_sampler(latent_config.flow_matching)
    stats = compute_latent_stats(dataset, batch_size=min(8, len(dataset)), num_workers=0)
    stats = {key: value.to(device) for key, value in stats.items()}
    context = batch["context"].to(device)
    future = batch["future"].to(device)
    latents_at_time, time_values, _, _ = flow.prepare_training_pair(future, latent_stats=stats)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        for _ in range(repeats):
            _ = model(latents_at_time, time_values, context)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "available": True,
        "device": str(device),
        "repeats": repeats,
        "seconds": elapsed,
        "forward_per_sec": repeats / elapsed if elapsed > 0 else np.nan,
        "context_shape": tuple(context.shape),
        "future_shape": tuple(future.shape),
        "cuda_peak_memory_mb": torch.cuda.max_memory_allocated() / (1024 * 1024) if device.type == "cuda" else None,
    }

pd.DataFrame([benchmark_model_forward(train_records)])


,available
0,False


In [9]:
def benchmark_flow_sampling(records, inference_steps=4):
    if not records:
        return {"available": False}
    dataset = FeatureLatentDataset(records)
    batch = next(iter(torch.utils.data.DataLoader(dataset, **make_loader_kwargs(1, 0, shuffle=False))))
    shape = infer_latent_dit_shape(dataset[0])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(latent_config.model, shape).to(device).eval()
    flow = build_flow_matching_sampler(latent_config.flow_matching)
    stats = compute_latent_stats(dataset, batch_size=min(8, len(dataset)), num_workers=0)
    stats = {key: value.to(device) for key, value in stats.items()}
    context = batch["context"].to(device)
    output_shape = tuple(batch["future"].shape)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    _ = flow.euler_sample(model, context, output_shape, latent_stats=stats, inference_steps=inference_steps)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "available": True,
        "device": str(device),
        "inference_steps": inference_steps,
        "seconds": elapsed,
        "seconds_per_step": elapsed / inference_steps,
        "cuda_peak_memory_mb": torch.cuda.max_memory_allocated() / (1024 * 1024) if device.type == "cuda" else None,
    }

pd.DataFrame([
    benchmark_flow_sampling(train_records, inference_steps=4),
    benchmark_flow_sampling(train_records, inference_steps=8),
])


,available
0,False
1,False


In [10]:
commands = [
    ["python", "entrypoints/preprocess.py", "--config", str(CONFIG_PATH), "--limit-per-split", "1", "--overwrite"],
    ["python", "entrypoints/extract_features.py", "--config", str(CONFIG_PATH), "--split", "all", "--stage", "all", "--limit-samples", "2", "--overwrite"],
    ["python", "entrypoints/train.py", "--config", str(CONFIG_PATH), "--run-id", RUN_ID, "--max-steps", "1", "--limit-samples", "2"],
    ["python", "entrypoints/inference.py", "--config", str(CONFIG_PATH), "--run-id", RUN_ID, "--limit-samples", "2", "--overwrite"],
]


def run_timed(command):
    start = time.perf_counter()
    result = subprocess.run(command, cwd=PROJECT_ROOT, capture_output=True, text=True, check=False)
    elapsed = time.perf_counter() - start
    return {
        "command": " ".join(command),
        "returncode": result.returncode,
        "seconds": elapsed,
        "stdout_tail": result.stdout[-500:],
        "stderr_tail": result.stderr[-500:],
    }

if RUN_SMOKE_COMMANDS:
    stage_timings = pd.DataFrame([run_timed(command) for command in commands])
else:
    stage_timings = pd.DataFrame({"command": [" ".join(command) for command in commands], "seconds": [None] * len(commands), "returncode": [None] * len(commands)})
stage_timings


,command,seconds,returncode
0,python entrypoints/preprocess.py --config /wor...,None,None
1,python entrypoints/extract_features.py --confi...,None,None
2,python entrypoints/train.py --config /workspac...,None,None
3,python entrypoints/inference.py --config /work...,None,None


In [11]:
bottlenecks = []
if not loader_benchmarks.empty and loader_benchmarks["samples_per_sec"].notna().any():
    best_loader = loader_benchmarks["samples_per_sec"].max()
    bottlenecks.append({"area": "feature tensor loading", "signal": f"best loader throughput: {best_loader:.2f} samples/sec"})

forward_info = benchmark_model_forward(train_records, repeats=2)
if forward_info.get("available"):
    bottlenecks.append({"area": "DiT forward", "signal": f"{forward_info['forward_per_sec']:.2f} forwards/sec on {forward_info['device']}"})

sampling_info = benchmark_flow_sampling(train_records, inference_steps=4)
if sampling_info.get("available"):
    bottlenecks.append({"area": "flow Euler sampling", "signal": f"{sampling_info['seconds_per_step']:.4f} sec/step; total scales roughly with flow_matching.inference_steps"})

if psutil is not None:
    memory = psutil.virtual_memory()
    bottlenecks.append({"area": "host memory", "signal": f"available {memory.available / (1024**3):.2f} GB / total {memory.total / (1024**3):.2f} GB"})

pd.DataFrame(bottlenecks)


,area,signal
0,host memory,available 52.65 GB / total 60.46 GB


## Bottleneck interpretation

- If feature tensor loading is slow, increase `training.num_workers`/`inference.num_workers`, use faster storage, or reduce duplicated `.pt` reads.
- If DiT forward is slow or memory-heavy, reduce `hidden_size`, `num_layers`, context `max_tokens`, or enable `gradient_checkpointing` for memory relief.
- If inference is slow, first reduce `flow_matching.inference_steps`; runtime scales nearly linearly with Euler steps.
- If feature extraction dominates, encoder/VAE inference is the bottleneck; keep `offline_cached` inference so VAE is not repeated during scoring.
